In [8]:
import pandas as pd
import requests

for year in range(2022, 2025):
    url = f"https://api.jolpi.ca/ergast/f1/{year}/races.json"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    races = data["MRData"]["RaceTable"]["Races"]
    df = pd.json_normalize(races)

    reduced_df = df[[
        "season",
        "round",
        "raceName",
        "Circuit.circuitName",
        "Circuit.Location.locality",
        "Circuit.Location.country",
        "date",
        "time",
    ]]

    reduced_df.to_json(
        path_or_buf=f"../data/raw/races/{year}_races.json",
        orient="records",
    )

In [ ]:
import pandas as pd
import requests

for year in range(2022, 2025):
    races = pd.read_json(f"../data/raw/races/{year}_races.json")
    rounds = len(races["round"].unique())
    
    print(f"{year} :total rounds: {rounds}")

    df_final = pd.DataFrame()

    for race in range(1, rounds + 1):
        url = f"https://api.jolpi.ca/ergast/f1/{year}/{race}/results.json"
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        races = data["MRData"]["RaceTable"]["Races"]

        df = pd.json_normalize(
            races,
            record_path="Results",
            meta=["season", "round"],
        )

        reduced_df = df[[
            "season",
            "round",
            "position",
            "points",
            "grid",
            "status",
            "Driver.driverId",
            "Constructor.constructorId",
        ]]

        df_final = pd.concat([df_final, reduced_df])

    df_final.to_json(
        path_or_buf=f"../data/raw/results/{year}_results.json",
        orient="records",
    )

2022 :total rounds: 22
2023 :total rounds: 22
2024 :total rounds: 24


HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2024/21/results.json

In [9]:
import pandas as pd

df = pd.DataFrame()
for year in [2022, 2023, 2024]:
    races = pd.read_json(f"../data/raw/races/{year}_races.json")
    results = pd.read_json(f"../data/raw/results/{year}_results.json")

    tmp_df = results.merge(races, on=["season", "round"], how="left")
    df = pd.concat([df, tmp_df])


df["position"] = pd.to_numeric(df["position"], errors="coerce")
df["grid"] = pd.to_numeric(df["grid"], errors="coerce")
df["round"] = pd.to_numeric(df["round"])
df["race_datetime"] = pd.to_datetime(df["date"].astype(str) + " " + df["time"])

df = df.sort_values(["Driver.driverId", "season", "round"])
df["driver_last_race_position"] = (
    df.groupby("Driver.driverId")["position"]
    .shift(1)
    .astype("Int64")
)
df["driver_median_position_last_3_races"] = (
    df.groupby("Driver.driverId")["position"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).median())
)

team_race = (
    df.groupby(["Constructor.constructorId", "season", "round"], as_index=False)["position"]
    .median()
    .rename(columns={"position": "constructor_race_position"})
)
team_race = team_race.sort_values(["Constructor.constructorId", "season", "round"])
team_race["constructor_median_last_3_races"] = (
    team_race.groupby("Constructor.constructorId")["constructor_race_position"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).median())
)

df = df.merge(
    team_race[["Constructor.constructorId", "season", "round", "constructor_median_last_3_races"]],
    on=["Constructor.constructorId", "season", "round"],
    how="left",
)

team_grid = (
    df.groupby(["Constructor.constructorId", "season", "round"])["grid"]
    .median()
    .reset_index(name="constructor_grid_median")
)
df = df.merge(team_grid, on=["Constructor.constructorId", "season", "round"])
df["constructor_median_last_3_races"] = (
    df["constructor_median_last_3_races"].fillna(df["constructor_grid_median"])
)

df = df.dropna(subset=["position", "grid"])
df = df.drop(columns=["date", "time"])

# e.g. "assume they finish where they start" when no history
df["driver_last_race_position"] = df["driver_last_race_position"].fillna(df["grid"])
df["driver_median_position_last_3_races"] = df["driver_median_position_last_3_races"].fillna(df["grid"])

df = df.rename(columns={
    "Driver.driverId": "driver_id",
    "Constructor.constructorId": "constructor_id",
    "Circuit.circuitName": "circuit",
    "Circuit.Location.locality": "locality",
    "Circuit.Location.country": "country",
})

df.to_json(
    path_or_buf=f"../data/processed/full_seasons.json",
    orient="records",
    date_format="iso",
)

In [18]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np
from sklearn.preprocessing import OrdinalEncoder


df = pd.read_json(f"../data/processed/full_seasons.json")
HALF_2024_ROUND = 12  # rounds 1–12 train, 13–24 test

train = df[(df["season"] < 2024) | ((df["season"] == 2024) & (df["round"] <= HALF_2024_ROUND))]
test  = df[(df["season"] == 2024) & (df["round"] > HALF_2024_ROUND)]

features = [
    "grid", 
    "driver_last_race_position", 
    "driver_median_position_last_3_races", 
    "driver_id", 
    "constructor_id",
    "constructor_median_last_3_races",
    "constructor_grid_median",
    "circuit", 
    "country"]

x_train= train[features]
y_train = train["position"]

x_test = test[features]
y_test = test["position"]

# Baseline finishing position = grid position
baseline_model = mean_absolute_error(y_test, x_test["grid"])
print(f"Baseline MAE (predict position = grid): {baseline_model:.2f}")


preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore",), ["driver_id", "constructor_id", "circuit"]),
    ("num", "passthrough", ["grid", "driver_last_race_position", "driver_median_position_last_3_races", "constructor_median_last_3_races", "constructor_grid_median"]),
])

model = Pipeline([
    ("prep", preprocessor),
    ("reg", RandomForestRegressor(n_estimators=500, random_state=42)),
])
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

print(f"Model MAE: {mean_absolute_error(y_test, y_pred):.2f}")

reprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ["driver_id", "constructor_id", "circuit"]),
    ("num", "passthrough", ["grid", "driver_last_race_position", "driver_median_position_last_3_races", "constructor_median_last_3_races", "constructor_grid_median"]),
])

model_2 = Pipeline([
    ("prep", reprocessor),
    ("reg", HistGradientBoostingRegressor(
        categorical_features=[0, 1, 2],  # indices of cat columns after transform
        random_state=42
    )),
])

model_2.fit(x_train, y_train)
y_pred = model_2.predict(x_test)

print(f"Model MAE: {mean_absolute_error(y_test, y_pred):.2f}")


Baseline MAE (predict position = grid): 3.24
Model MAE: 3.22
Model MAE: 3.51
